# Native vertical velocity within fitted eddy cores

Explore selected eddy-days before choosing one vertical-velocity metric for a population analysis. The NetCDF `w` is upward velocity (m/s); the fitted profile column `w` is relative vorticity (s⁻¹). Per-depth fitted centres and ellipses define the sampling region.


In [1]:
from pathlib import Path
import sys
import netCDF4 as nc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
if HERE.name != 'vertical_velocity':
    raise RuntimeError('Launch this notebook from the vertical_velocity folder')
sys.path.insert(0, str(HERE.parent))
sys.path.insert(0, str(HERE.parent / 'case_studies' / 'eddy_cross_sections_3d'))
import seacofs_tilt_tools as tilt
from case_section_tools import resolve_model_file
from vertical_velocity_tools import sample_snapshot, column_extrema


In [2]:
# Edit these controls after looking at the available IDs and days below.
CASES = [(6,1471)]  # e.g. [(123, 4567), (321, 4568)]
FRACTIONS = (0.5, 1.0, 1.5)
MAX_DEPTH_M = 1000
MAX_MISMATCH_M = 75
MAP_DEPTHS_M = (50, 200, 500)
MODEL_ROOT = Path('/srv/scratch/z3533156/26year_BRAN2020')


In [3]:
surface, _ = tilt.load_tilt_tables()
vertical = tilt.load_vert()
grid = tilt.load_grid()
print(f'{len(surface):,} surface eddy-days; {len(vertical):,} fitted depth rows')
display(surface[['Eddy','Day','Cyc','TiltDis','fname']].head(15))
print('Example well-resolved eddy-days:')
counts = vertical.loc[vertical.Depth.le(MAX_DEPTH_M)].groupby(['Eddy','Day']).Depth.nunique().sort_values(ascending=False)
display(counts.head(20).rename('n_fitted_depths').reset_index().merge(surface[['Eddy','Day','Cyc','TiltDis']],on=['Eddy','Day'],how='left'))


127,426 surface eddy-days; 2,824,609 fitted depth rows


,Eddy,Day,Cyc,TiltDis,fname
0,1,1462,CE,NaN,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
1,1,1463,CE,NaN,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
2,1,1464,CE,NaN,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
3,1,1465,CE,19.823350,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
4,1,1466,CE,15.055658,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
5,1,1467,CE,8.821740,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
6,1,1468,CE,5.770986,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
7,1,1469,CE,4.014082,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
8,1,1470,CE,2.294209,/srv/scratch/z3533156/26year_BRAN2020/outer_av...
9,1,1471,CE,1.540758,/srv/scratch/z3533156/26year_BRAN2020/outer_av...


Example well-resolved eddy-days:


,Eddy,Day,n_fitted_depths,Cyc,TiltDis
0,2982,10650,22,CE,NaN
1,1,1462,22,CE,NaN
2,1,1463,22,CE,NaN
3,1,1464,22,CE,NaN
4,1,1465,22,CE,19.823350
5,1,1466,22,CE,15.055658
6,1,1467,22,CE,8.821740
7,1,1468,22,CE,5.770986
8,1,1470,22,CE,2.294209
9,1,1471,22,CE,1.540758


Set `CASES` above, then run the remaining cells. Each `(Eddy, Day)` must have a surface row and a fitted vertical profile. Source files are read one snapshot at a time.


In [4]:
if not CASES:
    raise ValueError('Set CASES in the controls cell first')
all_depth, all_column, all_maps = [], [], {}
for eddy, day in CASES:
    s = surface.loc[surface.Eddy.eq(eddy) & surface.Day.eq(day)]
    profile = vertical.loc[vertical.Eddy.eq(eddy) & vertical.Day.eq(day)].sort_values('Depth')
    if len(s) != 1 or profile.empty:
        raise ValueError(f'Expected one surface row and a profile for {(eddy,day)}')
    path = resolve_model_file(s.iloc[0], MODEL_ROOT)
    print(f'{(eddy,day)}: {path.name}, {len(profile)} fitted levels')
    with nc.Dataset(path) as ds:
        depth, maps = sample_snapshot(ds, profile.iloc[0], profile.Depth.unique(), grid,
            fractions=FRACTIONS, max_depth_m=MAX_DEPTH_M, max_mismatch_m=MAX_MISMATCH_M)
    depth['TiltDis_km'] = s.iloc[0].TiltDis
    all_depth.append(depth)
    all_column.append(column_extrema(depth))
    all_maps[(eddy,day)] = maps
per_depth = pd.concat(all_depth, ignore_index=True)
per_column = pd.concat(all_column, ignore_index=True)
display(per_column)
display(per_depth[['Eddy','Day','fraction','Depth','n_core','n_valid','coverage','w_min','w_max','w_mean','z_min_m','z_max_m']])


(6, 1471): outer_avg_01461.nc, 26 fitted levels


ValueError: Model w and analysis grid have different horizontal shapes

In [ ]:
# Positive is upward. Compare extrema and mean as functions of fitted depth.
for (eddy, day), group in per_depth.groupby(['Eddy','Day']):
    fig, axes = plt.subplots(1, 2, figsize=(11,6), sharey=True)
    for fraction, g in group.groupby('fraction'):
        g = g.sort_values('Depth')
        axes[0].plot(1e3*g.w_max, g.Depth, 'o-', label=f'{fraction:g} core')
        axes[0].plot(1e3*g.w_min, g.Depth, 'o--', color=axes[0].lines[-1].get_color())
        axes[1].plot(g.coverage, g.Depth, 'o-', label=f'{fraction:g} core')
    axes[0].axvline(0,color='0.5',lw=0.8)
    axes[0].set(xlabel='Native w (mm/s): max solid, min dashed', ylabel='Fitted depth (m)')
    axes[1].set(xlabel='Valid core-cell fraction', xlim=(0,1.05))
    axes[0].invert_yaxis()
    axes[0].legend()
    axes[1].legend()
    fig.suptitle(f'Eddy {eddy}, day {day}; tilt {group.TiltDis_km.iloc[0]:.1f} km')
    fig.tight_layout()
    plt.show()


In [ ]:
# Local maps: show the full-core footprint at a few available fitted depths.
for (eddy, day), maps in all_maps.items():
    levels = sorted({d for f,d in maps if f == 1.0})
    chosen = [min(levels, key=lambda d:abs(d-target)) for target in MAP_DEPTHS_M] if levels else []
    chosen = list(dict.fromkeys(chosen))
    if not chosen: continue
    vmax = max(np.nanmax(np.abs(maps[(1.0,d)][0])) for d in chosen)
    fig, axes = plt.subplots(1,len(chosen),figsize=(5*len(chosen),4),squeeze=False)
    for ax,d in zip(axes[0],chosen):
        field,(i0,i1,j0,j1) = maps[(1.0,d)]
        img=ax.pcolormesh(grid.x_grid[i0:i1],grid.y_grid[j0:j1],(1e3*field).T,
                          cmap='RdBu_r',vmin=-1e3*vmax,vmax=1e3*vmax,shading='auto')
        ax.set(title=f'Fitted depth {d:.0f} m',xlabel='x (km)',ylabel='y (km)',aspect='equal')
        fig.colorbar(img,ax=ax,label='Upward w (mm/s)')
    fig.suptitle(f'Eddy {eddy}, day {day}; full core')
    fig.tight_layout()
    plt.show()


## Choosing a metric

Inspect whether extrema recur at similar depths and locations, whether they remain stable as the core fraction changes, and whether coverage is adequate. A minimum and maximum should be retained separately: averaging signed extrema can conceal a strong upwelling/downwelling dipole. For a later tilt correlation, compare these extrema with robust alternatives such as upper/lower core percentiles or area means, then bootstrap whole eddy tracks.
